# MAGFiLO EdgeAttNet v1

Add Data → `filament-segmentation-2026`. Clone this repo into `/kaggle/working` or Add Dataset (zip without `.venv`). GPU on. Smoke first, then train.


In [ ]:
import os, sys, subprocess
from pathlib import Path

try:
    from kaggle_secrets import UserSecretsClient

    sec = UserSecretsClient()
    for name in ("GIT_URL", "GITHUB_TOKEN", "DAGSHUB_TOKEN"):
        try:
            os.environ[name] = sec.get_secret(name)
        except Exception:
            pass
except Exception:
    pass

REPO = Path("/kaggle/working/filament-segmentation-2026")
url = os.environ.get("GIT_URL", "")
if url and not (REPO / "run.py").exists():
    token = os.environ.get("GITHUB_TOKEN", "")
    if token and url.startswith("https://"):
        url = url.replace("https://", f"https://{token}@", 1)
    subprocess.check_call(["git", "clone", "--depth", "1", url, str(REPO)])

root = (
    REPO
    if (REPO / "run.py").exists()
    else next(p for p in Path("/kaggle/input").rglob("run.py")).parent
)
sys.path.insert(0, str(root))
os.chdir(root)
print(root)

In [ ]:
!pip -q install python-dotenv mlflow dagshub pycocotools

In [ ]:
dagshub_token = os.environ.get("DAGSHUB_TOKEN", "")
if dagshub_token:
    Path(".env").write_text(
        "DAGSHUB_REPO_OWNER=BraveCactus\n"
        "DAGSHUB_REPO_NAME=solar-filament-segmentation\n"
        "DAGSHUB_EXPERIMENT_NAME=Test_Experiment\n"
        f"DAGSHUB_TOKEN={dagshub_token}\n",
        encoding="utf-8",
    )

In [ ]:
!python run.py check

In [ ]:
!python run.py edgeattn train --arch edgeattn --base 8 --size 128 --epochs 1 --batch 2 --max-steps 20 --eval-samples-val 5 --workers 0 --run-name edgeattn_smoke

In [ ]:
# !python run.py edgeattn train --arch unet --size 512 --epochs 8 --batch 8 --eval-samples-val 50 --eval-samples-train 50 --eval-tiled --workers 2 --run-name edgeattn_unet_512

In [ ]:
# !python run.py edgeattn train --arch edgeattn --size 512 --epochs 8 --batch 8 --eval-samples-val 50 --eval-samples-train 50 --eval-tiled --workers 2 --run-name edgeattn_eg_512

In [ ]:
# !python run.py edgeattn infer --weights /kaggle/working/outputs/edgeattn/experiments/edgeattn_eg_512/best.pt --tiled --tile-size 512 --tile-overlap 128 --submit